# Notebook preparing ModelArchive deposit

This notebook contains the necessary code to prepare the deposit of the protein structure models to ModelArchive. 

#### General settings, imports, variables and environments

Conda environment: viral_madepo

Created with `conda create -n viral_act_madepo python=3.10`. Then installed `jupyter notebook`, `pandas` through conda (command `conda install`).

In [1]:
!conda list --explicit

# This file may be used to create an environment using:
# $ conda create --name <env> --file <this file>
# platform: win-64
# created-by: conda 24.11.3
@EXPLICIT
https://conda.anaconda.org/conda-forge/noarch/ca-certificates-2026.2.25-h4c7d964_0.conda
https://conda.anaconda.org/conda-forge/noarch/python_abi-3.10-8_cp310.conda
https://conda.anaconda.org/conda-forge/noarch/tzdata-2025c-hc9c84f9_1.conda
https://conda.anaconda.org/conda-forge/win-64/ucrt-10.0.26100.0-h57928b3_0.conda
https://conda.anaconda.org/conda-forge/win-64/winpty-0.4.3-4.tar.bz2
https://conda.anaconda.org/conda-forge/win-64/libwinpthread-12.0.0.r4.gg4f2fc60ca-h57928b3_10.conda
https://conda.anaconda.org/conda-forge/win-64/vcomp14-14.44.35208-h818238b_34.conda
https://conda.anaconda.org/conda-forge/win-64/vc14_runtime-14.44.35208-h818238b_34.conda
https://conda.anaconda.org/conda-forge/win-64/vc-14.3-h41ae7f8_34.conda
https://conda.anaconda.org/conda-forge/win-64/bzip2-1.0.8-h0ad9c76_9.conda
https://conda.anaconda.org/

In [105]:
# imports 
import csv
import glob
import os
import shutil

import numpy as np
import pandas as pd

from pathlib import Path

In [3]:
# clear reference to different directories
pipeline_search_dir = os.getcwd()
master_dir = os.path.abspath(os.path.join(pipeline_search_dir, os.pardir, os.pardir))

In [38]:
# creating a directory for all the data we will generate
os.mkdir(os.path.join(pipeline_search_dir, "modelarchive"))

#### Copying the relevant structure prediction files

As a first step, we will copy the necessary structure prediction files for ModelArchive to a dedicted folder.

We will first do this for the 'annotated acetyltransferase' protein subcategory. The easiest way to get an overview of the 'succesful' structure predictions is to start from the protein structures we used as input for the FoldSeek search.

In [39]:
os.mkdir(os.path.join(pipeline_search_dir, "modelarchive", "annotated"))

In [40]:
# first, list all the files in the foldseek_in directory
files = os.listdir(os.path.join(pipeline_search_dir, "b_assess_annotated", "structure_comparison", "foldseek_in"))
# then filter for protein structure files and extract the protein identifiers (ncbi unique identifiers, ncbi_uids)
ncbi_uids = list()
not_relaxed = list()
for file in files:
    if ".pdb" in file:
        ncbi_uids.append("_".join(file.split("_")[0:3]))
        if "relaxed" not in file:
            not_relaxed.append("_".join(file.split("_")[0:3]))


In [41]:
# let's now use these ncbi_uids to copy the relevant files to the modelarchive folder
for uid in ncbi_uids:
    folder = os.path.join(pipeline_search_dir, "b_assess_annotated", "structure_prediction", "results", uid)
    # irrespective of succesful relaxation, the log and .json can be copied
    shutil.copyfile(os.path.join(folder, f"{uid}_info.log"), 
                    os.path.join(pipeline_search_dir, "modelarchive", "annotated", f"{uid}_info.log"))
    json_file = next(Path(folder).glob("*.json"))
    shutil.copyfile(json_file, os.path.join(pipeline_search_dir, "modelarchive", "annotated", json_file.name))
    if uid not in not_relaxed:
        shutil.copyfile(os.path.join(folder, f"{uid}_relaxed.pdb"), 
                            os.path.join(pipeline_search_dir, "modelarchive", "annotated", f"{uid}_relaxed.pdb"))
    if uid in not_relaxed:
        pdb = f"{uid}_unrelaxed_rank_1_model_[12345]_ptmx1.pdb"
        for file in glob.glob(os.path.join(folder, pdb)):
            shutil.copyfile(file,
                            os.path.join(pipeline_search_dir, "modelarchive", "annotated", os.path.basename(file)))


Let's now do the same for the 'proteins of unknown function':

In [42]:
os.mkdir(os.path.join(pipeline_search_dir, "modelarchive", "unknown"))

In [43]:
# first, list all the files in the foldseek_in directory
files = os.listdir(os.path.join(pipeline_search_dir, "c_assess_unknown", "structure_comparison", "foldseek_in"))
# then filter for protein structure files and extract the protein identifiers (ncbi unique identifiers, ncbi_uids)
ncbi_uids = list()
not_relaxed = list()
for file in files:
    if ".pdb" in file:
        ncbi_uids.append("_".join(file.split("_")[0:3]))
        if "relaxed" not in file:
            not_relaxed.append("_".join(file.split("_")[0:3]))


In [44]:
# let's now use these ncbi_uids to copy the relevant files to the modelarchive folder
for uid in ncbi_uids:
    folder = os.path.join(pipeline_search_dir, "c_assess_unknown", "structure_prediction", "results", uid)
    # irrespective of succesful relaxation, the log and .json can be copied
    shutil.copyfile(os.path.join(folder, f"{uid}_info.log"), 
                    os.path.join(pipeline_search_dir, "modelarchive", "unknown", f"{uid}_info.log"))
    json_file = next(Path(folder).glob("*.json"))
    shutil.copyfile(json_file, os.path.join(pipeline_search_dir, "modelarchive", "unknown", json_file.name))
    if uid not in not_relaxed:
        shutil.copyfile(os.path.join(folder, f"{uid}_relaxed.pdb"), 
                            os.path.join(pipeline_search_dir, "modelarchive", "unknown", f"{uid}_relaxed.pdb"))
    if uid in not_relaxed:
        pdb = f"{uid}_unrelaxed_rank_1_model_[12345]_ptmx1.pdb"
        for file in glob.glob(os.path.join(folder, pdb)):
            shutil.copyfile(file,
                            os.path.join(pipeline_search_dir, "modelarchive", "unknown", os.path.basename(file)))


Let's finally also add in the elongated proteins:

In [46]:
os.mkdir(os.path.join(pipeline_search_dir, "modelarchive", "elongated"))

In [48]:
for uid in os.listdir(os.path.join(pipeline_search_dir, "b_assess_annotated", "investigate_early_starts", "structure_prediction", "results")):
    folder = os.path.join(pipeline_search_dir, "b_assess_annotated", "investigate_early_starts", "structure_prediction", "results", uid)
    shutil.copyfile(os.path.join(folder, f"{uid}_info.log"), 
                    os.path.join(pipeline_search_dir, "modelarchive", "elongated", f"{uid}_info.log"))
    json_file = next(Path(folder).glob("*.json"))
    shutil.copyfile(json_file, os.path.join(pipeline_search_dir, "modelarchive", "elongated", json_file.name))
    shutil.copyfile(os.path.join(folder, f"{uid}_relaxed.pdb"), 
                    os.path.join(pipeline_search_dir, "modelarchive", "elongated", f"{uid}_relaxed.pdb"))


#### Summarizing all relevant metadata

Let's now parse these files and extract the data relevant to the ModelArchive deposit.

In [ ]:
# reading in protein information obtained earlier in the project
protein_data_unknown = pd.read_csv(os.path.join(pipeline_search_dir, "a_input", "protein_overview", "input_search.tsv"), sep="\t")
protein_data_ann = pd.read_csv(os.path.join(pipeline_search_dir, "b_assess_annotated", "annotated_acetyltransferases_seq_matched.tsv"), sep="\t")

In [ ]:
# create a file for all the metadata from the annotated acetyltransferase subset
ma_descr_ann = os.path.join(pipeline_search_dir, "modelarchive", "annotated", "metadata_annotated.tsv")
with open(ma_descr_ann, "w", newline = "") as out:
    writer = csv.writer(out, delimiter = "\t")
    # write header
    writer.writerow(["NCBI unique identifier", "UniProt identifier", "model title", "model abstract",
        "name structure file", "name json file", "name log file", "relaxed", "MMSeqs2 API call date", "pLDDT", "pTM"])
    # loop over the entries and extract the relevant information
    for file in os.listdir(os.path.join(pipeline_search_dir, "modelarchive", "annotated")):
        if file.endswith(".log"):
            # extract protein name used by me and based on that, NCBI unique identifier
            protein = "_".join(file.split("_")[0:-1])
            ncbi_uid = int(protein.split("_")[-1])
            uniprot_id = ""
            # extract MMSeqs2 API call date, pLDDT and pTM from the log file
            with open(os.path.join(pipeline_search_dir, "modelarchive", "annotated", file)) as f:
                for line in f:
                    if "Starting MSA search" in line:
                        date = line.split()[0]
                    if "rank_1_model_" in line:
                        plddt = float(line.split()[-2].split("=")[-1].removesuffix(","))
                        ptm = float(line.split()[-1].split("=")[-1])
            # check if the relaxed structure file is present, if not, use the unrelaxed one and note that in the metadata
            if os.path.exists(os.path.join(pipeline_search_dir, "modelarchive", "annotated", f"{protein}_relaxed.pdb")):
                relaxed = True
                structure_file = f"{protein}_relaxed.pdb"
            else:
                relaxed = False
                structure_file = os.path.basename(glob.glob(os.path.join(pipeline_search_dir, "modelarchive", "annotated", f"{protein}_unrelaxed_rank_1_model_[12345]_ptmx1.pdb"))[0])
            # retrieve name of json file containing PAE values
            json_file = os.path.basename(glob.glob(os.path.join(pipeline_search_dir, "modelarchive", "annotated", f"{protein}_rank_1_model_[12345]_ptmx1.json"))[0])
            # get protein description used in model title and abstract based on stored metadata
            if ncbi_uid in protein_data_ann["ncbi_protein_uid"].values:
                ncbi_protein_name = protein_data_ann[protein_data_ann["ncbi_protein_uid"] == ncbi_uid]["ncbi_protein_name"].values[0]
                descrip_title = f"Model for {ncbi_protein_name} (NCBI unique identifier: {ncbi_uid}"
                # obtain UniProt identifiers (only stored if UniProt annotation hinted at acetyltransferase function)
                if pd.isna(protein_data_ann[protein_data_ann["ncbi_protein_uid"] == ncbi_uid]["uniprot_protein_id"].values[0]) != True:
                    uniprot_id = protein_data_ann[protein_data_ann["ncbi_protein_uid"] == ncbi_uid]["uniprot_protein_id"].values[0]
                    descrip_title += f", UniProt identifier: {uniprot_id}"
                descrip_title += ")"
            elif ncbi_uid in protein_data_unknown["ncbi_protein_uid"].values:
                ncbi_protein_name = protein_data_unknown[protein_data_unknown["ncbi_protein_uid"] == ncbi_uid]["ncbi_protein_name"].values[0]
                descrip_title = f"Model for {ncbi_protein_name} (NCBI unique identifier: {ncbi_uid})"
            descrip_abstract = "AlphaFold2 m" + descrip_title[1:] + f". This protein was part of the 'annotated acetyltransferases' subset and is referred to by the authors as {protein}."
            # actually write all metadata to file
            writer.writerow([ncbi_uid, uniprot_id, descrip_title, descrip_abstract, structure_file, json_file, file, relaxed, date, plddt, ptm])

In [ ]:
# create a file for all the metadata from the proteins of unknown function subset
ma_descr_unknown = os.path.join(pipeline_search_dir, "modelarchive", "unknown", "metadata_unknown.tsv")
with open(ma_descr_unknown, "w", newline = "") as out:
    writer = csv.writer(out, delimiter = "\t")
    # write header
    writer.writerow(["NCBI unique identifier", "UniProt identifier", "model title", "model abstract",
        "name structure file", "name json file", "name log file", "relaxed", "MMSeqs2 API call date", "pLDDT", "pTM"])
    # loop over the entries and extract the relevant information
    for file in os.listdir(os.path.join(pipeline_search_dir, "modelarchive", "unknown")):
        if file.endswith(".log"):
            # extract protein name used by me and based on that, NCBI unique identifier
            protein = "_".join(file.split("_")[0:-1])
            ncbi_uid = int(protein.split("_")[-1])
            uniprot_id = ""
            # extract MMSeqs2 API call date, pLDDT and pTM from the log file
            with open(os.path.join(pipeline_search_dir, "modelarchive", "unknown", file)) as f:
                for line in f:
                    if "Starting MSA search" in line:
                        date = line.split()[0]
                    if "rank_1_model_" in line:
                        plddt = float(line.split()[-2].split("=")[-1].removesuffix(","))
                        ptm = float(line.split()[-1].split("=")[-1])
            # check if the relaxed structure file is present, if not, use the unrelaxed one and note that in the metadata
            if os.path.exists(os.path.join(pipeline_search_dir, "modelarchive", "unknown", f"{protein}_relaxed.pdb")):
                relaxed = True
                structure_file = f"{protein}_relaxed.pdb"
            else:
                relaxed = False
                structure_file = os.path.basename(glob.glob(os.path.join(pipeline_search_dir, "modelarchive", "unknown", f"{protein}_unrelaxed_rank_1_model_[12345]_ptmx1.pdb"))[0])
            # retrieve name of json file containing PAE values
            json_file = os.path.basename(glob.glob(os.path.join(pipeline_search_dir, "modelarchive", "unknown", f"{protein}_rank_1_model_[12345]_ptmx1.json"))[0])
            # get protein description used in model title and abstract based on stored metadata
            if ncbi_uid in protein_data_ann["ncbi_protein_uid"].values:
                ncbi_protein_name = protein_data_ann[protein_data_ann["ncbi_protein_uid"] == ncbi_uid]["ncbi_protein_name"].values[0]
                descrip_title = f"Model for {ncbi_protein_name} (NCBI unique identifier: {ncbi_uid}"
                # obtain UniProt identifiers (only stored if UniProt annotation hinted at acetyltransferase function)
                if pd.isna(protein_data_ann[protein_data_ann["ncbi_protein_uid"] == ncbi_uid]["uniprot_protein_id"].values[0]) != True:
                    uniprot_id = protein_data_ann[protein_data_ann["ncbi_protein_uid"] == ncbi_uid]["uniprot_protein_id"].values[0]
                    descrip_title += f", UniProt identifier: {uniprot_id}"
                descrip_title += ")"
            elif ncbi_uid in protein_data_unknown["ncbi_protein_uid"].values:
                ncbi_protein_name = protein_data_unknown[protein_data_unknown["ncbi_protein_uid"] == ncbi_uid]["ncbi_protein_name"].values[0]
                descrip_title = f"Model for {ncbi_protein_name} (NCBI unique identifier: {ncbi_uid})"
            descrip_abstract = "AlphaFold2 m" + descrip_title[1:] + f". This protein was part of the 'proteins of unknown function' subset and is referred to by the authors as {protein}."
            # actually write all metadata to file
            writer.writerow([ncbi_uid, uniprot_id, descrip_title, descrip_abstract, structure_file, json_file, file, relaxed, date, plddt, ptm])

In [ ]:
# create a file for all the metadata from the elongated subset
ma_descr_elongated = os.path.join(pipeline_search_dir, "modelarchive", "elongated", "metadata_elongated.tsv")
with open(ma_descr_elongated, "w", newline = "") as out:
    writer = csv.writer(out, delimiter = "\t")
    # write header
    writer.writerow(["NCBI unique identifier", "UniProt identifier", "model title", "model abstract",
        "name structure file", "name json file", "name log file", "relaxed", "MMSeqs2 API call date", "pLDDT", "pTM"])
    # loop over the entries and extract the relevant information
    for file in os.listdir(os.path.join(pipeline_search_dir, "modelarchive", "elongated")):
        if file.endswith(".log"):
            # extract protein name used by me and based on that, NCBI unique identifier
            protein = "_".join(file.split("_")[0:-1])
            ncbi_uid = int(protein.split("_")[-2])
            uniprot_id = ""
            # extract MMSeqs2 API call date, pLDDT and pTM from the log file
            with open(os.path.join(pipeline_search_dir, "modelarchive", "elongated", file)) as f:
                for line in f:
                    if "Starting MSA search" in line:
                        date = line.split()[0]
                    if "rank_1_model_" in line:
                        plddt = float(line.split()[-2].split("=")[-1].removesuffix(","))
                        ptm = float(line.split()[-1].split("=")[-1])
            # check if the relaxed structure file is present, if not, use the unrelaxed one and note that in the metadata
            if os.path.exists(os.path.join(pipeline_search_dir, "modelarchive", "elongated", f"{protein}_relaxed.pdb")):
                relaxed = True
                structure_file = f"{protein}_relaxed.pdb"
            else:
                relaxed = False
                structure_file = os.path.basename(glob.glob(os.path.join(pipeline_search_dir, "modelarchive", "elongated", f"{protein}_unrelaxed_rank_1_model_[12345]_ptmx1.pdb"))[0])
            # retrieve name of json file containing PAE values
            json_file = os.path.basename(glob.glob(os.path.join(pipeline_search_dir, "modelarchive", "elongated", f"{protein}_rank_1_model_[12345]_ptmx1.json"))[0])
            # get protein description used in model title and abstract based on stored metadata
            if ncbi_uid in protein_data_ann["ncbi_protein_uid"].values:
                ncbi_protein_name = protein_data_ann[protein_data_ann["ncbi_protein_uid"] == ncbi_uid]["ncbi_protein_name"].values[0]
                descrip_title = f"Model for upstream in-frame extended variant of {ncbi_protein_name} (NCBI unique identifier of not-extended version: {ncbi_uid}"
                # obtain UniProt identifiers (only stored if UniProt annotation hinted at acetyltransferase function)
                if pd.isna(protein_data_ann[protein_data_ann["ncbi_protein_uid"] == ncbi_uid]["uniprot_protein_id"].values[0]) != True:
                    uniprot_id = protein_data_ann[protein_data_ann["ncbi_protein_uid"] == ncbi_uid]["uniprot_protein_id"].values[0]
                    descrip_title += f", UniProt identifier of not-extended version: {uniprot_id}"
                descrip_title += ")"
            elif ncbi_uid in protein_data_unknown["ncbi_protein_uid"].values:
                ncbi_protein_name = protein_data_unknown[protein_data_unknown["ncbi_protein_uid"] == ncbi_uid]["ncbi_protein_name"].values[0]
                descrip_title = f"Model for upstream in-frame extended variant of{ncbi_protein_name} (NCBI unique identifier of not-extended version: {ncbi_uid})"
            descrip_abstract = "AlphaFold2 m" + descrip_title[1:] + f". This protein was part of the 'annotated acetyltransferases' subset and is referred to by the authors as {protein}. For details on the in-frame upstream extension, please consult the manuscript."
            # actually write all metadata to file
            writer.writerow(["", "", descrip_title, descrip_abstract, structure_file, json_file, file, relaxed, date, plddt, ptm])

Now everything should be combined, and we should make sure that for the duplicate entries we keep the best quality models. (Four proteins belonging to the 'annotated acetyltransferase' subset that were in the end not deemed a plausible acetyltransferase, were then reconsidered in the 'proteins of unknown function' subset and thus had their structures predicted twice).

In [116]:
# to get the duplicate ones, let's simply find the duplicates in the list of ncbi_uids
    # we will just reuse the existing code to extract those uids
uids_unknown = list()
files_unknown = os.listdir(os.path.join(pipeline_search_dir, "c_assess_unknown", "structure_comparison", "foldseek_in"))
for file in files_unknown:
    if ".pdb" in file:
        uids_unknown.append("_".join(file.split("_")[0:3]))
files_annotated = os.listdir(os.path.join(pipeline_search_dir, "b_assess_annotated", "structure_comparison", "foldseek_in"))
uids_annotated = list()
for file in files_annotated:
    if ".pdb" in file:
        uids_annotated.append("_".join(file.split("_")[0:3]))
    # find the duplicates
duplicates = set(uids_unknown) & set(uids_annotated)

In [119]:
# let's now check which structure prediction result was better for the duplicate entries
    # read in the metadata files to get the pLDDT and pTM values for the duplicate entries
metadata_unknown = pd.read_csv(os.path.join(pipeline_search_dir, "modelarchive", "unknown", "metadata_unknown.tsv"), sep="\t")
metadata_annotated = pd.read_csv(os.path.join(pipeline_search_dir, "modelarchive", "annotated", "metadata_annotated.tsv"), sep="\t")
for dup in duplicates:
    plddt_unknown = metadata_unknown[metadata_unknown["name log file"].str.contains(dup)]["pLDDT"].values[0]
    ptm_unknown = metadata_unknown[metadata_unknown["name log file"].str.contains(dup)]["pTM"].values[0]
    plddt_annotated = metadata_annotated[metadata_annotated["name log file"].str.contains(dup)]["pLDDT"].values[0]
    ptm_annotated = metadata_annotated[metadata_annotated["name log file"].str.contains(dup)]["pTM"].values[0]
    # based on these values, move the better one to the modelarchive folder and note in the metadata which one was better (if one is better in both metrics, if one is better in pLDDT but not in pTM, or if they are similar enough that it depends on the metric which one is better)
    if (plddt_unknown > plddt_annotated) & (ptm_unknown > ptm_annotated):
        print(f"{dup} better in unknown")
        shutil.move(os.path.join(pipeline_search_dir, "modelarchive", "unknown", f"{dup}_info.log"),
                        os.path.join(pipeline_search_dir, "modelarchive",  f"{dup}_info.log"))
        json_file = os.path.basename(glob.glob(os.path.join(pipeline_search_dir, "modelarchive", "unknown", f"{dup}_rank_1_model_[12345]_ptmx1.json"))[0])
        shutil.move(os.path.join(pipeline_search_dir, "modelarchive", "unknown", json_file),
                        os.path.join(pipeline_search_dir, "modelarchive",  json_file))
        if metadata_unknown[metadata_unknown["name log file"].str.contains(dup)]["relaxed"].values[0] == True:
            shutil.move(os.path.join(pipeline_search_dir, "modelarchive", "unknown", f"{dup}_relaxed.pdb"),
                            os.path.join(pipeline_search_dir, "modelarchive",  f"{dup}_relaxed.pdb"))
    elif (plddt_annotated > plddt_unknown) & (ptm_annotated > ptm_unknown):
        print(f"{dup} better in annotated")
        shutil.move(os.path.join(pipeline_search_dir, "modelarchive", "annotated", f"{dup}_info.log"),
                        os.path.join(pipeline_search_dir, "modelarchive",  f"{dup}_info.log"))
        json_file = os.path.basename(glob.glob(os.path.join(pipeline_search_dir, "modelarchive", "annotated", f"{dup}_rank_1_model_[12345]_ptmx1.json"))[0])
        shutil.move(os.path.join(pipeline_search_dir, "modelarchive", "annotated", json_file),
                        os.path.join(pipeline_search_dir, "modelarchive",  json_file))
        if metadata_annotated[metadata_annotated["name log file"].str.contains(dup)]["relaxed"].values[0] == True:
            shutil.move(os.path.join(pipeline_search_dir, "modelarchive", "annotated", f"{dup}_relaxed.pdb"),
                            os.path.join(pipeline_search_dir, "modelarchive",  f"{dup}_relaxed.pdb"))
    else:
        print(f"{dup} depend on metric")

ncbi_uid_952977516 better in unknown
ncbi_uid_383389569 better in annotated
ncbi_uid_2289974540 better in unknown
ncbi_uid_1930266517 better in annotated


We can now move everything else, and remove the lower quality duplicates. 

In [ ]:
# elongated ones are all unique, so we can just move them to the modelarchive folder
for file in os.listdir(os.path.join(pipeline_search_dir, "modelarchive", "elongated")):
    shutil.move(os.path.join(pipeline_search_dir, "modelarchive", "elongated", file),
                os.path.join(pipeline_search_dir, "modelarchive", file))
# now let's move the non-duplicate ones to the modelarchive folder and remove the duplicate ones (as they are already moved based on which one is better)
for file in os.listdir(os.path.join(pipeline_search_dir, "modelarchive", "annotated")):
    is_duplicate = any(dup in file for dup in duplicates)
    if not is_duplicate:
        shutil.move(os.path.join(pipeline_search_dir, "modelarchive", "annotated", file),
                    os.path.join(pipeline_search_dir, "modelarchive", file))
    if is_duplicate:
        os.remove(os.path.join(pipeline_search_dir, "modelarchive", "annotated", file))
for file in os.listdir(os.path.join(pipeline_search_dir, "modelarchive", "unknown")):
    is_duplicate = any(dup in file for dup in duplicates)
    if not is_duplicate:
        shutil.move(os.path.join(pipeline_search_dir, "modelarchive", "unknown", file),
                    os.path.join(pipeline_search_dir, "modelarchive", file))
    if is_duplicate:
        os.remove(os.path.join(pipeline_search_dir, "modelarchive", "unknown", file))

In [127]:
# after inspection, we can safely remove the empty directories
os.rmdir(os.path.join(pipeline_search_dir, "modelarchive", "elongated"))
os.rmdir(os.path.join(pipeline_search_dir, "modelarchive", "annotated"))
os.rmdir(os.path.join(pipeline_search_dir, "modelarchive", "unknown"))

PermissionError: [WinError 5] Toegang geweigerd: 'c:\\Users\\hanne\\OneDrive\\Documenten\\GitHub\\ACES-AcT\\pipeline\\1_search\\modelarchive\\elongated'

Finally, the metadata will be merged, and phiKMV Map homolog will be clearly labeled and put first. 

In [145]:
# read in the metadata files, append to each other and remove duplicates
metadata_annotated = pd.read_csv(os.path.join(pipeline_search_dir, "modelarchive", "metadata_annotated.tsv"), sep="\t")
metadata_unknown = pd.read_csv(os.path.join(pipeline_search_dir, "modelarchive", "metadata_unknown.tsv"), sep="\t")
metadata_elongated = pd.read_csv(os.path.join(pipeline_search_dir, "modelarchive", "metadata_elongated.tsv"), sep="\t")
metadata_all = pd.concat([metadata_annotated, metadata_unknown, metadata_elongated], ignore_index=True)
metadata_all["NCBI unique identifier"] = metadata_all["NCBI unique identifier"].astype("Int64")
metadata_cleaned = (metadata_all.sort_values(["pLDDT", "pTM"], ascending = False)
                    .drop_duplicates(subset=["name structure file"])
                    .sort_index().reset_index(drop=True))
metadata_cleaned.to_csv(os.path.join(pipeline_search_dir, "modelarchive", "metadata.tsv"), sep = "\t", index = False)